In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, HTML
from rouge_score import rouge_scorer
from torch.optim import Adam
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
import pandas as pd
import json
from tqdm.auto import tqdm

tqdm.pandas()
import plotly.graph_objects as go
import numpy as np
import scipy.stats

from math_utils import *
from math_wrong_dataset import build_math_wrong_dataset, MathWrongDataset
from frank_wolfe_optimizer import FrankWolfeOptimizer


model_name = "Qwen/Qwen3-1.7B"
tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer = tok
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype=torch.bfloat16,  # 建议用 bf16 匹配 Qwen 训练格式
    trust_remote_code=True,
)
model.requires_grad_(False)
# 提取词表全集用于寻找最近邻 (Input Space Interrogation)
all_embeddings = model.get_input_embeddings().weight.detach()

dst = build_math_wrong_dataset("/workspace/yiqiuguo/lsrl/qwen3-1.7b_math-500_rollout8_len32768_final.jsonl", tok, thinking_ratio=0.8, max_samples=1000)

d = dst[75]
question_text = d["question_text"]
answer_text = d["answer_text"]
parts = answer_text.split("</think>")
thinking_text = parts[0]
connector_text = "</think>" + parts[1].split("\\boxed{")[0] + "\\boxed{"
fast_connector_text = "</think>" "The final answer is \n$$\n" + "\\boxed{"
gt_text = d["gt_text"]+"\n"
pred_text = d["pred_text"]+"\n"


# 获取各部分 Embeddings (假设变量 question_text, thinking_text 等已定义)
def get_embeds(text):
    ids = tok.encode(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    return model.get_input_embeddings()(ids).detach(), ids


# ==========================================
# 1. 变量初始化与准备
# ==========================================
embeds_q, ids_q = get_embeds(question_text)
embeds_think, ids_think = get_embeds(thinking_text)
embeds_end_think, ids_end_think = get_embeds("</think>")
embeds_conn, ids_conn = get_embeds(connector_text)
embeds_fast_conn, ids_fast_conn = get_embeds(fast_connector_text)
embeds_gt, ids_gt = get_embeds(gt_text)
embeds_pred, ids_pred = get_embeds(pred_text)

target_latent = embeds_think.detach().clone().float()
target_latent.requires_grad = True


def visualize_token_gradients(ids_think, grad_norm, tokenizer):
    """
    ids_think: torch.Tensor [4183]
    grad_norm: torch.Tensor [4183] (或者 list/numpy)
    tokenizer: 对应的分词器 (需支持 decode 方法)
    """

    # 1. 数据预处理
    tokens = [tokenizer.decode([tid]) for tid in ids_think]
    if isinstance(grad_norm, torch.Tensor):
        grads = grad_norm.detach().cpu().numpy()
    else:
        grads = grad_norm
    # 2. 归一化梯度以便映射颜色 (0-1)
    g_min, g_max = grads.min(), grads.max()
    # 防止除以 0
    norm_grads = (grads - g_min) / (g_max - g_min + 1e-9)

    # 3. 构建 HTML 字符串
    # 这里的逻辑：背景色从 rgb(200, 200, 255) [淡蓝] 到 rgb(255, 100, 100) [红]
    html_content = """
    <div style="font-family: monospace; line-height: 2.0; padding: 20px; background-color: #f5f5f5; border-radius: 10px;">
    """

    for i, (token, score) in enumerate(zip(tokens, norm_grads)):
        # 颜色映射逻辑：无色(白色) -> 红色
        # score = 0 (低梯度) -> rgb(255, 255, 255) [白色]
        # score = 1 (高梯度) -> rgb(255, 0, 0) [纯红]
        
        r = 255
        g = int(255 * (1 - score))
        b = int(255 * (1 - score))
        
        # 处理空格和换行
        display_token = f"[{i+1}]: " + token.replace("\n", "<br>").replace(" ", "&nbsp;")
        
        # 只有当 score 较大时才使用白色字体，否则黑色
        text_color = 'white' if score > 0.6 else 'black'
        
        html_content += f"""
        <span style="background-color: rgb({r}, {g}, {b}); 
                     padding: 0px 2px; 
                     border-radius: 2px;
                     color: {text_color};" 
              title="Grad Norm: {grads[i]:.4f}">
            {display_token}
        </span>"""

    html_content += "</div>"

    # 4. 在 Jupyter 中渲染
    display(HTML(html_content))

def top_k_overlap(g1, g2, k_ratio=0.1):
    """计算梯度最大的前 k_ratio 比例的 token 的重合度"""
    k = max(1, int(len(g1) * k_ratio))
    
    # 获取 top-k 的索引
    topk_idx1 = set(torch.topk(g1, k).indices.cpu().numpy())
    topk_idx2 = set(torch.topk(g2, k).indices.cpu().numpy())
    
    # 计算交集大小
    overlap_count = len(topk_idx1.intersection(topk_idx2))
    overlap_ratio = overlap_count / k
    
    return overlap_ratio


target_latent.grad = None
orig_full = torch.cat([embeds_q, target_latent, embeds_conn, embeds_pred], dim=1).to(model.dtype).to(model.device)
orig_out = model(inputs_embeds=orig_full)
target_logits = orig_out.logits[:, embeds_q.shape[1] + target_latent.shape[1] - 1 : -1, :].view(-1, orig_out.logits.shape[-1])
labels = torch.cat([ids_conn, ids_pred], dim=1).view(-1)
loss = F.cross_entropy(target_logits, labels)
print(loss)
loss.backward()
grad1 = target_latent.grad.detach().clone()
grad_norm = torch.norm(target_latent.grad*target_latent, dim=2, p=1).flatten()
print(grad_norm.mean().item())

go.Figure(go.Bar(x=np.arange(1, len(grad_norm) + 1), y=grad_norm.detach().cpu().numpy(), marker_color="red")).show()
visualize_token_gradients(ids_think.flatten(), grad_norm, tokenizer)

target_latent.grad = None
orig_full = torch.cat([embeds_q, target_latent, embeds_fast_conn, embeds_gt], dim=1).to(model.dtype).to(model.device)
orig_out = model(inputs_embeds=orig_full)
target_logits = orig_out.logits[:, embeds_q.shape[1] + target_latent.shape[1] + embeds_fast_conn.shape[1]- 1 : -1, :].view(-1, orig_out.logits.shape[-1])
labels = torch.cat([ids_gt], dim=1).view(-1)
loss = F.cross_entropy(target_logits, labels)
print(loss)
loss.backward()
grad2 = target_latent.grad.detach().clone()
grad_norm = torch.norm(target_latent.grad*target_latent, dim=2, p=1).flatten()
print(grad_norm.mean().item())

go.Figure(go.Bar(x=np.arange(1, len(grad_norm) + 1), y=grad_norm.detach().cpu().numpy(), marker_color="red")).show()
visualize_token_gradients(ids_think.flatten(), grad_norm, tokenizer)

# # 确保梯度清零
# if target_latent.grad is not None:
#     target_latent.grad.zero_()

# # ==========================================
# # 1. 单独计算 GT 的梯度
# # ==========================================
# input_gt = torch.cat([embeds_q, target_latent, embeds_fast_conn, embeds_gt], dim=1).to(model.dtype)
# out_gt = model(inputs_embeds=input_gt)
# shift_logits_gt = out_gt.logits[:, embeds_q.shape[1] + target_latent.shape[1] + embeds_fast_conn.shape[1] - 1 : -1, :].contiguous()
# loss_gt = F.cross_entropy(shift_logits_gt.view(-1, shift_logits_gt.shape[-1]), ids_gt.view(-1), reduction='mean')

# loss_gt.backward() # 注意：这里不要影响原来的计算图
# grad_gt = target_latent.grad.detach().clone()

# target_latent.grad.zero_() # 必须清零，准备算 Pred 的梯度

# # ==========================================
# # 2. 单独计算 Pred 的梯度
# # ==========================================
# input_pred = torch.cat([embeds_q, target_latent, embeds_conn, embeds_pred], dim=1).to(model.dtype)
# out_pred = model(inputs_embeds=input_pred)
# shift_logits_pred = out_pred.logits[:, embeds_q.shape[1] + target_latent.shape[1] + embeds_conn.shape[1] - 1 : -1, :].contiguous()
# loss_pred = F.cross_entropy(shift_logits_pred.view(-1, shift_logits_pred.shape[-1]), ids_pred.view(-1), reduction='mean')

# loss_pred.backward()
# grad_pred = target_latent.grad.detach().clone()

# # ==========================================
# # 3. 核心：在 L2 归一化空间计算对比梯度
# # ==========================================
# # 沿着最后一个维度 (embedding dim) 进行 L2 归一化，使得每个 token 的梯度向量长度为 1
# norm_grad_gt = F.normalize(grad_gt, p=2, dim=-1)
# norm_grad_pred = F.normalize(grad_pred, p=2, dim=-1)

# # 计算归一化后的对比梯度
# # 这里不仅保留了 GT 的方向，还强烈地去除了 Pred 的方向
# grad_contrastive_normalized = norm_grad_gt - norm_grad_pred

# # 计算最终用于可视化的对比范数 (L1 或 L2 皆可)
# grad_norm_contrastive = torch.norm(grad_contrastive_normalized, dim=2, p=1).flatten()
# saliency_score = torch.sum(grad_contrastive_normalized * target_latent.detach(), dim=-1).abs().flatten()
# scores = grad_norm_contrastive.cpu().numpy()

# # 找到 70% 分位数的阈值
# threshold = np.percentile(scores, 90) 

# # 低于阈值的直接设为 0，高于阈值的重新缩放到 0-1
# cleaned_scores = np.where(scores > threshold, scores - threshold, 0)
# max_score = cleaned_scores.max()
# if max_score > 0:
#     cleaned_scores = cleaned_scores / max_score
# # 可视化
# go.Figure(go.Bar(x=np.arange(1, len(grad_norm_contrastive) + 1), y=grad_norm_contrastive.cpu().numpy(), marker_color="purple")).show()
# go.Figure(go.Bar(x=np.arange(1, len(saliency_score) + 1), y=saliency_score.cpu().numpy(), marker_color="purple")).show()
# go.Figure(go.Bar(x=np.arange(1, len(cleaned_scores) + 1), y=cleaned_scores, marker_color="purple")).show()
# visualize_token_gradients(ids_think.flatten(), cleaned_scores, tokenizer)

In [ ]:
inp = tokenizer(question_text+thinking_text+connector_text,return_tensors='pt')['input_ids'].to(model.device)
out= model.generate(inp,max_new_tokens=10)
print(tokenizer.decode(out[0][inp.shape[-1]:]))
# print(question_text+thinking_text+connector_text)
print(pred_text)
print(ids_pred[0])
print(tokenizer.batch_decode(ids_pred[0]))
print(out[0][inp.shape[-1]:])
print(tokenizer.batch_decode(out[0][inp.shape[-1]:]))

In [ ]:
inp = tokenizer(question_text+thinking_text+fast_connector_text,return_tensors='pt')['input_ids'].to(model.device)
out= model.generate(inp,max_new_tokens=10)
print(tokenizer.decode(out[0][inp.shape[-1]:]))

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import ipywidgets as widgets
from IPython.display import display, HTML
import plotly.graph_objects as go
import numpy as np
import math

# ==========================================
# 0. 定义严格的 COLD Soft NLL Loss
# ==========================================
def soft_nll(logits_perturbed, logits):
    """
    logits_perturbed: 大模型对下一步的预测 (target_lm_logits)
    logits: 正在被优化的连续软序列 (opt_logits)
    """
    p = F.softmax(logits_perturbed, dim=-1)
    logp = F.log_softmax(logits, dim=-1)
    return -(p * logp).sum(dim=-1).mean()

# ==========================================
# 1. 初始化模型与配置
# ==========================================
model_id = "Qwen/Qwen2.5-1.5B-Instruct" 
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model {model_id} on {device}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    torch_dtype=torch.float16,
    trust_remote_code=True
)
model.eval()

for param in model.parameters():
    param.requires_grad = False

embed_layer = model.get_input_embeddings()

# ==========================================
# 2. 任务数据与超参设置
# ==========================================
question = "Solve the equation: 3x + 5 = 14"
ground_truth = "3"
budget_tokens = 95

prefix_text = f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\nLet's think step by step.\n"
suffix_text = f"\n\nThe final answer is \\boxed{{{ground_truth}}}<|im_end|>"

prefix_ids = tokenizer(prefix_text, return_tensors="pt").input_ids.to(device)
suffix_ids = tokenizer(suffix_text, return_tensors="pt").input_ids.to(device)

steps = 2000
lr = 0.05
initial_noise_std = 0.01 
lambda_lm = 2.0          
lambda_gt = 1.0          
temp = 1.0               
TOP_K = 2
eval_every = 100

# ==========================================
# 3. UI 组件初始化
# ==========================================
fig = go.FigureWidget()
fig.add_scatter(y=[], name='Soft LM Loss (Continuous)', mode='lines')
fig.add_scatter(y=[], name='GT Loss (Coherence)', mode='lines')
fig.add_scatter(y=[], name='Total Energy (Loss)', mode='lines')
fig.add_scatter(x=[], y=[], name='True Discrete LM Loss', mode='lines+markers', line=dict(dash='dot', color='purple'))

fig.layout.title = "COLD Optimization Energy Landscape & True Loss Probe"
fig.layout.height = 400
fig.layout.margin = dict(l=20, r=20, t=40, b=20)

html_panel_soft = widgets.HTML(value="<h3>Initializing...</h3>")
html_panel_discrete = widgets.HTML(value="")
step_slider = widgets.IntSlider(min=0, max=0, value=0, description='History Step:', layout=widgets.Layout(width='80%'))

history_tokens = []
history_kl = []
history_discrete_data = {} # 新增：保存包含 tokens 和对应概率的字典
history_discrete_loss = {} 
history_losses = {'lm': [], 'gt': [], 'total': []}
history_true_lm = {'steps': [], 'losses': []} 

def render_html(step_idx):
    if not history_tokens: return
    
    # 1. 渲染 Soft Logits
    tokens = history_tokens[step_idx]
    kl_scores = history_kl[step_idx]
    html_str_soft = "<h4>1. Soft Logits Argmax (Continuous State)</h4><div style='display:flex; flex-wrap:wrap; font-family:monospace; line-height:1.5;'>"
    for t, kl in zip(tokens, kl_scores):
        intensity = min(1.0, float(kl) / 3.0) 
        bg_color = f"rgba(255, 50, 50, {intensity})"
        t_clean = t.replace('<', '&lt;').replace('>', '&gt;')
        html_str_soft += f"<div style='background-color:{bg_color}; padding:2px 6px; margin:2px; border:1px solid #ddd; border-radius:4px;'>{t_clean}</div>"
    html_str_soft += "</div>"
    html_panel_soft.value = html_str_soft

    # 2. 渲染 Discrete Generation & Suffix Tokens (按真实概率着色)
    discrete_step = step_idx - (step_idx % eval_every) 
    if discrete_step in history_discrete_data:
        d_data = history_discrete_data[discrete_step]
        d_tokens = d_data['tokens']
        d_probs = d_data['probs']
        true_loss = history_discrete_loss.get(discrete_step, 0.0)
        
        d_html = f"<h4>2. Discrete Top-{TOP_K} Generation + Suffix (Updated at Step {discrete_step})</h4>"
        d_html += f"<b style='color:purple; font-size:16px;'>True Discrete LM Loss (Cross-Entropy): {true_loss:.4f}</b><br/>"
        d_html += f"<span style='font-size:12px; color:#666;'>* Background color indicates conditional probability (Redder = Lower Probability). Hover for exact values.</span><br/><br/>"
        d_html += "<div style='display:flex; flex-wrap:wrap; font-family:monospace; line-height:1.5;'>"
        
        for t, prob in zip(d_tokens, d_probs):
            # 概率越低，颜色越红 (1.0 - prob)
            intensity = 1.0 - prob
            # 使用透明度来实现红色深浅，底色纯红
            bg_color = f"rgba(255, 50, 50, {intensity})"
            t_clean = t.replace('<', '&lt;').replace('>', '&gt;')
            # 添加 title 属性，鼠标悬停可以显示具体概率
            d_html += f"<div title='Prob: {prob:.4f}' style='background-color:{bg_color}; padding:2px 6px; margin:2px; border:1px solid #ddd; border-radius:4px;'>{t_clean}</div>"
        d_html += "</div>"
        html_panel_discrete.value = d_html

def on_slider_change(change):
    render_html(change.new)

step_slider.observe(on_slider_change, names='value')
display(fig, step_slider, widgets.VBox([html_panel_soft, html_panel_discrete]))

# ==========================================
# 4. 离散化自回归生成函数 
# ==========================================
def generate_with_guidance(model, prefix_ids, opt_logits, top_k=10):
    model.eval()
    generated_ids = []
    current_input_ids = prefix_ids
    past_key_values = None
    
    with torch.no_grad():
        for t in range(opt_logits.shape[1]): 
            outputs = model(input_ids=current_input_ids, past_key_values=past_key_values, use_cache=True)
            past_key_values = outputs.past_key_values
            
            next_token_logits = outputs.logits[0, -1, :]
            _, top_k_indices = torch.topk(next_token_logits, top_k)
            
            current_opt_logits = opt_logits[0, t, :].clone()
            masked_logits = torch.full_like(current_opt_logits, float('-inf'))
            masked_logits[top_k_indices] = current_opt_logits[top_k_indices]
            
            selected_token_id = masked_logits.argmax().unsqueeze(0).unsqueeze(0)
            generated_ids.append(selected_token_id.item())
            current_input_ids = selected_token_id
            
    return generated_ids

# ==========================================
# 5. 初始化
# ==========================================
with torch.no_grad():
    init_output = model.generate(
        prefix_ids, 
        max_new_tokens=budget_tokens, 
        min_new_tokens=budget_tokens, 
        do_sample=False,
        output_logits=True,
        return_dict_in_generate=True,
        pad_token_id=tokenizer.eos_token_id
    )
    init_logits = torch.stack(init_output.logits, dim=1).to(torch.float32)

opt_logits = init_logits.clone().detach().requires_grad_(True)
p_step0 = F.softmax(opt_logits.detach() / temp, dim=-1)
optimizer = torch.optim.Adam([opt_logits], lr=lr)

# ==========================================
# 6. Langevin Dynamics 核心优化循环
# ==========================================
prefix_embeds = embed_layer(prefix_ids)
suffix_embeds = embed_layer(suffix_ids)

for step in range(steps):
    current_noise_std = initial_noise_std * (1.0 - step / steps)
    
    log_soft_probs = F.log_softmax(opt_logits / temp, dim=-1)
    soft_probs = torch.exp(log_soft_probs).to(embed_layer.weight.dtype)
    soft_embeds = torch.matmul(soft_probs, embed_layer.weight)
    
    concat_embeds = torch.cat([prefix_embeds, soft_embeds, suffix_embeds], dim=1)
    outputs = model(inputs_embeds=concat_embeds)
    model_logits = outputs.logits.to(torch.float32) 
    
    idx_start_soft = prefix_ids.shape[1] - 1
    idx_end_soft = idx_start_soft + budget_tokens
    target_lm_logits = model_logits[:, idx_start_soft:idx_end_soft, :].detach()
    
    soft_lm_loss = soft_nll(target_lm_logits / temp, opt_logits / temp)
    
    idx_start_suffix = idx_end_soft - 1
    idx_end_suffix = idx_start_suffix + suffix_ids.shape[1]
    pred_suffix_logits = model_logits[:, idx_start_suffix:idx_end_suffix, :]
    gt_loss = F.cross_entropy(
        pred_suffix_logits.reshape(-1, pred_suffix_logits.size(-1)), 
        suffix_ids.reshape(-1)
    )
    
    total_loss = lambda_lm * soft_lm_loss + lambda_gt * gt_loss
    
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_([opt_logits], max_norm=1.0)
    optimizer.step()
    
    with torch.no_grad():
        noise = torch.randn_like(opt_logits) * current_noise_std
        opt_logits.data = opt_logits.data + noise
        
        current_p = torch.exp(log_soft_probs)
        kl_per_token = F.kl_div(p_step0.log(), current_p, reduction='none').sum(dim=-1).squeeze(0).cpu().numpy()
        argmax_ids = opt_logits.argmax(dim=-1).squeeze(0)
        current_tokens = [tokenizer.decode([tok_id]) for tok_id in argmax_ids]
        
        history_tokens.append(current_tokens)
        history_kl.append(kl_per_token)
        
        def safe_val(v): return v if not (math.isnan(v) or math.isinf(v)) else None
            
        history_losses['lm'].append(safe_val(soft_lm_loss.item()))
        history_losses['gt'].append(safe_val(gt_loss.item()))
        history_losses['total'].append(safe_val(total_loss.item()))
        
        if step % eval_every == 0 or step == steps - 1:
            # 1. 执行真实的引导生成
            generated_ids = generate_with_guidance(model, prefix_ids, opt_logits, top_k=TOP_K)
            
            # 2. 计算真实句子的离散 LM Loss 并提取每个 token 的概率
            gen_tensor = torch.tensor([generated_ids], device=device)
            full_ids = torch.cat([prefix_ids, gen_tensor, suffix_ids], dim=1)
            
            disc_out = model(input_ids=full_ids)
            # 取出只属于 generated 和 suffix 对应位置的 logits 和 labels
            shift_logits = disc_out.logits[0, prefix_ids.shape[1]-1 : -1, :]
            shift_labels = full_ids[0, prefix_ids.shape[1]:]
            
            true_lm_loss_val = F.cross_entropy(shift_logits, shift_labels).item()
            history_discrete_loss[step] = true_lm_loss_val
            
            history_true_lm['steps'].append(step)
            history_true_lm['losses'].append(true_lm_loss_val)
            
            # ===== 新增：提取概率与 Suffix 组合 =====
            # 计算预测出来的所有可能 token 的概率
            shift_probs = F.softmax(shift_logits, dim=-1)
            # 提取实际发生的 token (生成的内容 + Suffix) 被预测到的概率
            token_probs = shift_probs.gather(dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1).tolist()
            
            # 解码完整的 (Generated + Suffix) 序列
            gs_ids = shift_labels.tolist()
            gs_tokens = [tokenizer.decode([tid]) for tid in gs_ids]
            
            # 保存到历史记录中供渲染
            history_discrete_data[step] = {
                'tokens': gs_tokens,
                'probs': token_probs
            }
            # ========================================

            # 3. 更新前端
            step_slider.max = step
            step_slider.value = step
            render_html(step)
            
            x_steps = list(range(len(history_losses['lm'])))
            with fig.batch_update():
                fig.data[0].x = x_steps
                fig.data[0].y = history_losses['lm']
                fig.data[1].x = x_steps
                fig.data[1].y = history_losses['gt']
                fig.data[2].x = x_steps
                fig.data[2].y = history_losses['total']
                
                fig.data[3].x = history_true_lm['steps']
                fig.data[3].y = history_true_lm['losses']

html_panel_soft.value += "<br/><b>Optimization Complete! Use the slider to review the history.</b>"

In [ ]:
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
)
print(text)

In [ ]:
13678

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import ipywidgets as widgets
from IPython.display import display, HTML
import matplotlib.pyplot as plt

# ==========================================
# 1. 初始化模型与参数
# ==========================================
MODEL_NAME = "Qwen/Qwen3-0.6B-Base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()
for param in model.parameters():
    param.requires_grad = False

def top_p_filtering(logits, top_p=0.95):
    """标准的 Top-P (Nucleus) 采样过滤"""
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    
    sorted_indices_to_remove = cumulative_probs > top_p
    sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
    sorted_indices_to_remove[0] = 0
    
    indices_to_remove = sorted_indices[sorted_indices_to_remove]
    filtered_logits = logits.clone()
    filtered_logits[indices_to_remove] = -float('Inf')
    return filtered_logits

# ==========================================
# 2. 场景设定与核心超参数 (新增弹性缓冲)
# ==========================================
prefix_text = "Once upon a time, a little robot named Toby found a mysterious glowing stone in the forest."
suffix_text = "And from that day on, the robot never needed to recharge its battery again."

prefix_ids = tokenizer.encode(prefix_text, return_tensors="pt").to(device)
suffix_ids = tokenizer.encode(suffix_text, return_tensors="pt").to(device)

TOP_P = 0.95
PROB_THRESHOLD = 0.01
MAX_NEW_TOKENS = 1000
STOP_LOSS_THRESHOLD = 0.5 

# 【核心创新配置】：缓冲占位符 (Elastic Buffer)
EXPECTED_STEPS = 15 # 预期用多少步走到 Suffix (动态缓冲区的初始长度)
# 使用中性符号作为语义占位符，避免引入太强的语法偏见
NEUTRAL_TOKEN_ID = tokenizer.encode(".")[0] 

generated_ids = []
history_logs = []
loss_history = []

out_html = display(HTML("<h3>实时生成区:</h3><div id='gen_text' style='border:1px solid #ccc; padding:10px; font-size:16px;'></div>"), display_id=True)
current_text = prefix_text + " <span style='color:blue; font-weight:bold;'>"

print(f"\n[开始生成] Target Suffix: '{suffix_text}'\n")

# ==========================================
# 3. 核心生成循环 (Elastic Gradient-Guided Infilling)
# ==========================================
for step in range(MAX_NEW_TOKENS):
    current_input_ids = torch.cat([prefix_ids, torch.tensor([generated_ids]).to(device)], dim=1) if generated_ids else prefix_ids
    
    # --- 第1次 Forward：获取正常 LM 的预测分布 ---
    with torch.no_grad():
        outputs = model(current_input_ids)
        next_logits = outputs.logits[0, -1, :] 
    
    lm_probs = F.softmax(next_logits, dim=-1)
    filtered_logits = top_p_filtering(next_logits, top_p=TOP_P)
    
    temp_logits = filtered_logits.clone().detach().requires_grad_(True)
    soft_probs = F.softmax(temp_logits, dim=-1) 
    
    word_embeddings = model.get_input_embeddings()
    soft_embed = soft_probs @ word_embeddings.weight 
    
    # --- 【新增核心机制】：构建动态弹性缓冲垫 ---
    # 随着步数增加，缓冲区(gap)越来越短，直到 0
    gap_len = max(0, EXPECTED_STEPS - step)
    if gap_len > 0:
        buffer_ids = torch.tensor([[NEUTRAL_TOKEN_ID] * gap_len]).to(device)
        buffer_embeds = word_embeddings(buffer_ids)
    else:
        buffer_embeds = torch.empty((1, 0, word_embeddings.embedding_dim)).to(device)
        
    # --- 第2次 Forward：计算包含 Suffix 的 Loss ---
    past_embeds = word_embeddings(current_input_ids)      
    suffix_embeds = word_embeddings(suffix_ids)           
    
    # 物理拼接：[过去文本] + [当前软词 1 步] + [弹性缓冲 L 步] + [未来后缀]
    total_embeds = torch.cat([past_embeds, soft_embed.view(1, 1, -1), buffer_embeds, suffix_embeds], dim=1)
    outputs_with_suffix = model(inputs_embeds=total_embeds)
    
    # 提取 Suffix 部分的预测 Logits
    # Suffix 的预测位置偏移 = 过去长度 + 1(软词) + 缓冲长度 - 1(错位预测)
    offset = current_input_ids.shape[1] + gap_len
    suffix_len = suffix_ids.shape[1]
    
    shift_logits = outputs_with_suffix.logits[0, offset : offset + suffix_len, :]
    shift_labels = suffix_ids[0]
    
    loss = F.cross_entropy(shift_logits, shift_labels)
    loss_history.append(loss.item())
    
    if loss.item() < STOP_LOSS_THRESHOLD and gap_len == 0:
        print(f"\n[停止信号] 发现 Suffix Loss 极低 ({loss.item():.4f}) 且距离归零，完美过渡！")
        break
        
    # --- 第1次 Backward：获取远期向导梯度 ---
    loss.backward()
    neg_grads = -temp_logits.grad 
    
    # --- 筛选与融合决策 ---
    valid_mask = lm_probs > PROB_THRESHOLD
    valid_neg_grads = neg_grads.clone()
    valid_neg_grads[~valid_mask] = -float('inf')
    
    best_token_id = valid_neg_grads.argmax().item()
    if valid_neg_grads[best_token_id] == -float('inf'):
        best_token_id = lm_probs.argmax().item()
        
    generated_ids.append(best_token_id)
    
    # 实时渲染
    gen_token_str = tokenizer.decode([best_token_id])
    current_text += gen_token_str.replace('\n', '<br>')
    
    # UI 显示当前步数和 Buffer 长度
    buffer_ui = f"<span style='color:orange;'> [距目标 {gap_len} 步] </span>" if gap_len > 0 else "<span style='color:green;'> [对接中] </span>"
    out_html.update(HTML(f"<h3>实时生成区 (Step {step+1} | Suffix Loss: {loss.item():.4f}):</h3><div style='border:1px solid #ccc; padding:10px; font-size:16px;'>{current_text}</span> {buffer_ui} <span style='color:gray;'>{suffix_text}</span></div>"))
    
    # 收集日志
    top5_grad_vals, top5_grad_idx = torch.topk(neg_grads, 5)
    top5_grad_info = [(tokenizer.decode([idx.item()]), val.item()) for idx, val in zip(top5_grad_idx, top5_grad_vals)]
    
    top5_lm_probs, top5_lm_idx = torch.topk(lm_probs, 5)
    top5_lm_info = [(tokenizer.decode([idx.item()]), p.item(), neg_grads[idx].item()) for idx, p in zip(top5_lm_idx, top5_lm_probs)]
    
    history_logs.append({
        'step': step + 1,
        'loss': loss.item(),
        'gap': gap_len,
        'chosen_token': gen_token_str,
        'top5_grad': top5_grad_info,
        'top5_lm': top5_lm_info
    })

current_text += "</span>"
out_html.update(HTML(f"<h3>最终生成结果:</h3><div style='border:1px solid #ccc; padding:10px; font-size:16px;'>{current_text}</span> <span style='color:black; font-weight:bold;'>{suffix_text}</span></div>"))

print("\n" + "="*50)
print("生成分析图表与日志准备完毕。")
print("="*50)

# ==========================================
# 4. 可视化模块
# ==========================================

# 4.1 绘制全局 Suffix Loss 下降折线图
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(loss_history) + 1), loss_history, marker='o', linestyle='-', color='b', linewidth=2, markersize=6)
plt.title("Suffix Loss over Steps (with Elastic Buffer)", fontsize=14)
plt.xlabel("Generation Step", fontsize=12)
plt.ylabel("Loss (Lower is better)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(range(1, len(loss_history) + 1))
plt.tight_layout()
plt.show()

# 4.2 详细日志的交互式滑块
def view_step(step_idx):
    if not history_logs:
        return
    log = history_logs[step_idx - 1]
    
    html_content = f"<h4>Step {log['step']} | 缓冲垫长度: {log['gap']} | 当前 Suffix Loss: <span style='color:red;'>{log['loss']:.4f}</span></h4>"
    html_content += f"<p><b>最终选定词:</b> <span style='background-color:#d4edda; padding:2px 6px; border-radius:4px;'><b>{log['chosen_token']}</b></span></p>"
    
    html_content += "<div style='display:flex; gap:20px;'>"
    
    html_content += "<div style='flex:1;'>"
    html_content += "<b>📈 -Grad Top 5 (远期潜力最高):</b><br><table border='1' style='border-collapse: collapse; text-align: left; width:100%;'>"
    html_content += "<tr style='background-color:#f8f9fa;'><th>候选词</th><th>-Grad 值</th></tr>"
    for word, grad in log['top5_grad']:
        word_clean = word.replace('<', '&lt;').replace('>', '&gt;')
        html_content += f"<tr><td>'{word_clean}'</td><td>{grad:.6f}</td></tr>"
    html_content += "</table></div>"
    
    html_content += "<div style='flex:1;'>"
    html_content += "<b>🧠 LM Probs Top 5 (当下最通顺):</b><br><table border='1' style='border-collapse: collapse; text-align: left; width:100%;'>"
    html_content += "<tr style='background-color:#f8f9fa;'><th>候选词</th><th>LM 概率</th><th>其对应的 -Grad</th></tr>"
    for word, prob, grad in log['top5_lm']:
        word_clean = word.replace('<', '&lt;').replace('>', '&gt;')
        row_style = "background-color:#e2e3e5;" if word == log['chosen_token'] else ""
        html_content += f"<tr><td style='{row_style}'>'{word_clean}'</td><td>{prob*100:.2f}%</td><td>{grad:.6f}</td></tr>"
    html_content += "</table></div>"
    
    html_content += "</div>"
    display(HTML(html_content))

if history_logs:
    step_slider = widgets.IntSlider(min=1, max=len(history_logs), step=1, value=1, description='查看 Step:', layout=widgets.Layout(width='500px'))
    widgets.interact(view_step, step_idx=step_slider)

Loading model and tokenizer...



[开始生成] Target Suffix: 'And from that day on, the robot never needed to recharge its battery again.'



KeyboardInterrupt: 